# Group 2 — Telecom Customer Churn
## Detailed Data Understanding Notebook

**Lead:** Madu Miracle Kosi — Data Understanding Lead  
**Project:** Group 2 Telecom Customer Churn  
**Dataset:** `Customer_churn.csv`

### Purpose

This notebook creates the technical evidence for the Data Understanding stage. The project brief assigns Madu ownership of the dataset source, schema, variables, initial quality and raw-customer-data understanding, with a required handoff of the dataset summary, raw-data assessment and shared explanation of important customer fields.

The notebook deliberately stops before cleaning/model training. The workflow is:

**Understand → Inspect → Document → Handoff → Prepare**

The raw dataset used here contains **7,043 rows × 21 columns**.

## 0. Project Context

The project uses the IBM Telco Customer Churn sample/classroom dataset. The brief states that this classroom data must not be represented as real customer data from MTN, Airtel, Glo, 9mobile or another Nigerian telecom operator.

The team workflow requires the dataset to be understood and raw quality issues reviewed before cleaning and modelling begin.

**Notebook rule:** diagnostic transformations may be used to inspect data, but the raw source DataFrame is not overwritten during this stage.

## 1. Import pandas

In [ ]:
import pandas as pd  # Import pandas for reading and inspecting the tabular customer dataset.

### Narration

Pandas is sufficient for the initial Data Understanding stage. Machine-learning libraries are intentionally not introduced yet because modelling is a later project stage.

## 2. Load the Raw Dataset

In [ ]:
file_path = "/content/Customer_churn.csv"  # Define the Google Colab path to the uploaded raw CSV file.
df = pd.read_csv(file_path)  # Read the raw CSV into a pandas DataFrame without applying cleaning rules.

### Narration

The raw CSV is loaded exactly as supplied. In Colab, upload `Customer_churn.csv` to the session or mount the agreed Google Drive location and update `file_path` if necessary.

**Important:** keep the original raw file unchanged. Cleaned data should be created separately later by the Data Preparation Lead.

## 3. Confirm Dataset Shape

In [ ]:
rows, columns = df.shape  # Extract the number of observations and fields.
print("Number of rows:", rows)  # Report the customer-record count.
print("Number of columns:", columns)  # Report the field count.

### Narration

The supplied dataset contains **7,043 customer records and 21 columns**.

The team's Data Understanding Deliverable independently records the same dimensions and interprets one row as one customer record.

## 4. Inspect the First Five Records

In [ ]:
display(df.head())  # Display the first five raw records for an initial visual inspection.

### Narration

This is the first human-readable view of the customer matrix.

We look for the overall shape of one customer record, the identifier, the target, numeric fields, categorical fields and immediately visible anomalies.

This is only an initial visual inspection. Formal quality checks follow.

## 5. Inspect the Complete Schema

In [ ]:
for column in df.columns:  # Iterate through every field in the raw dataset.
    print(column)  # Print the current field name.

### Narration

The schema is the foundation of the data dictionary.

The team has grouped the fields into the target plus account/financial, service-subscription and demographic dimensions. The detailed deliverable documents the agreed meaning and observed values for all 21 fields.

## 6. Inspect Data Types and Structure

In [ ]:
df.info()  # Display row count, field names, non-null counts and raw pandas data types.

### Narration

This inspection reveals whether the raw storage type matches the conceptual meaning of each field.

A critical finding is that `TotalCharges` is stored as **object**, while `MonthlyCharges` is **float64**. The team's assessment identifies `TotalCharges` as an action-required datatype issue.

We do not fix that here. The purpose is to document the raw condition before preparation.

## 7. Numerical Descriptive Statistics

In [ ]:
display(df.describe())  # Summarise numerical columns currently recognised by pandas.

### Narration

The numerical summary gives initial ranges and scale.

`tenure` ranges from **0 to 72 months** in the supplied CSV.

`TotalCharges` does not appear as a normal numerical field because it is stored as text. That absence is itself evidence of a datatype issue.

## 8. Inspect the Target Variable

In [ ]:
target_counts = df["Churn"].value_counts(dropna=False)  # Count every observed Churn value, including any missing values.
target_percentages = df["Churn"].value_counts(normalize=True, dropna=False).mul(100).round(2)  # Calculate the percentage for each target class.
target_summary = pd.DataFrame({"Count": target_counts, "Percentage": target_percentages})  # Combine counts and percentages into one table.
display(target_summary)  # Display the target distribution.

### Narration

The target distribution is:

- `No`: **5,174 (73.46%)**
- `Yes`: **1,869 (26.54%)**

The classes are therefore imbalanced, with retained customers as the majority class. The team's deliverable records the same distribution and notes that later evaluation must not rely on accuracy alone.

## 9. Formal Missing-Value Check

In [ ]:
missing_by_column = df.isnull().sum()  # Count values that pandas explicitly recognises as missing.
display(missing_by_column.sort_values(ascending=False))  # Display missing counts from highest to lowest.
print("Total pandas-recognised missing values:", int(missing_by_column.sum()))  # Print the dataset-wide total.

### Narration

The raw dataset has **0 pandas-recognised null values**.

However, this does **not** mean the dataset contains no missing information. A blank can be stored as a string rather than as `NaN`. The `TotalCharges` investigation below demonstrates why both formal-null and content-based checks are necessary.

## 10. Detect Whitespace-Only `TotalCharges` Values

In [ ]:
totalcharges_blank_mask = df["TotalCharges"].astype(str).str.strip().eq("")  # Identify TotalCharges values that become empty after whitespace is removed.
blank_totalcharges_rows = df.loc[totalcharges_blank_mask]  # Select the affected customer records.
print("Whitespace-only TotalCharges records:", int(totalcharges_blank_mask.sum()))  # Count the affected records.
display(blank_totalcharges_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]])  # Display useful context for each affected record.

### Narration

This identifies **11 whitespace-only values** in `TotalCharges`.

This explains why `isnull()` returned zero for this issue: the cells contain whitespace strings, not pandas `NaN` values.

The team's completed raw-data assessment identifies exactly 11 such records and states that they require explicit treatment before numeric conversion.

## 11. Reveal Invisible Characters with `repr()`

In [ ]:
blank_totalcharges_values = df.loc[totalcharges_blank_mask, "TotalCharges"].map(repr)  # Convert affected raw values into visible Python representations.
display(blank_totalcharges_values)  # Display the exact raw representation of each whitespace value.

### Narration

`repr()` is useful because a normal DataFrame display can make whitespace look like an empty cell.

This gives direct evidence that the values are strings containing whitespace rather than formal nulls.

## 12. Investigate the Pattern of the 11 Records

In [ ]:
blank_tenure = df.loc[totalcharges_blank_mask, "tenure"].value_counts(dropna=False)  # Count tenure values among affected customers.
blank_churn = df.loc[totalcharges_blank_mask, "Churn"].value_counts(dropna=False)  # Count Churn values among affected customers.
blank_contract = df.loc[totalcharges_blank_mask, "Contract"].value_counts(dropna=False)  # Count contract types among affected customers.
print("Tenure pattern:")  # Label the tenure result.
print(blank_tenure)  # Display the tenure pattern.
print("Churn pattern:")  # Label the Churn result.
print(blank_churn)  # Display the Churn pattern.
print("Contract pattern:")  # Label the contract result.
print(blank_contract)  # Display the contract pattern.

### Narration

The pattern found in the supplied CSV is:

- All **11** affected records have `tenure = 0`.
- All **11** affected records have `Churn = No`.
- Each has a valid `MonthlyCharges` value.
- The contract types are not all identical.

The team's deliverable records the same pattern.

### Interpretation

The pattern **suggests** these customers are at the beginning of their service relationship and therefore have no recorded cumulative charge value yet.

However, the raw CSV contains no explicit billing-event timestamp that independently proves the exact reason. Therefore:

> **Observation:** all 11 have tenure 0 and Churn No.  
> **Interpretation:** this may represent new customers without accumulated charges.  
> **Decision:** treatment belongs to Data Preparation and must be documented.

Do not silently replace these values during Data Understanding.

## 13. Test Numeric Conversion Without Modifying the Raw Column

In [ ]:
totalcharges_numeric_test = pd.to_numeric(df["TotalCharges"], errors="coerce")  # Temporarily attempt numeric conversion and coerce invalid strings only in this diagnostic Series.
conversion_failures = totalcharges_numeric_test.isna()  # Identify records that failed the temporary conversion.
print("Values failing numeric conversion:", int(conversion_failures.sum()))  # Count conversion failures.
display(df.loc[conversion_failures, ["customerID", "tenure", "TotalCharges", "Churn"]])  # Display the records that caused conversion failures.

### Narration

The diagnostic conversion produces **11 failures**, confirming that the whitespace entries are not valid numerical values.

The original `TotalCharges` column remains untouched.

The next stage should standardise whitespace, convert to numeric and explicitly decide how the 11 tenure-zero records are handled. The team's technical data contract specifies this approach.

## 14. Check Exact Duplicate Rows

In [ ]:
duplicate_row_mask = df.duplicated(keep=False)  # Identify rows that are exact duplicates of another row.
duplicate_row_count = int(df.duplicated().sum())  # Count duplicate occurrences using pandas' default duplicate logic.
print("Number of duplicate rows:", duplicate_row_count)  # Report the duplicate count.
display(df.loc[duplicate_row_mask].head(20))  # Show examples if duplicates exist.

### Narration

The supplied raw dataset contains **0 exact duplicate rows**.

The team's validation snapshot records the same result.

No evidence currently supports removing duplicate rows. The result should still be retained as part of the raw-data quality baseline.

## 15. Check `customerID` Uniqueness

In [ ]:
customer_id_duplicate_count = int(df["customerID"].duplicated().sum())  # Count repeated customer IDs.
unique_customer_count = int(df["customerID"].nunique())  # Count distinct customer IDs.
print("Unique customer IDs:", unique_customer_count)  # Report the number of unique IDs.
print("Repeated customer ID occurrences:", customer_id_duplicate_count)  # Report repeated IDs.

### Narration

There are **7,043 unique customer IDs** for **7,043 records**, with **0 repeated ID occurrences**.

Therefore `customerID` acts as a unique identifier in this extract. It is useful for traceability but should not normally be treated as a predictive feature because it identifies the customer rather than describing behaviour.

## 16. Inspect Raw Categorical Values

In [ ]:
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()  # Collect raw string/object columns.
for column in categorical_columns:  # Iterate through each raw categorical field.
    print(f"\n--- {column} ---")  # Print a heading for the current field.
    print(df[column].unique())  # Display the distinct raw values.

### Narration

This establishes the raw category vocabulary before encoding.

Some service fields contain structural values such as `No internet service` and `No phone service`. These are not automatically equivalent to ordinary `No` because they describe the absence of the underlying service.

The team's agreed field definitions explicitly preserve these structural categories.

Encoding is deliberately deferred to Data Preparation.

## 17. Inspect Numeric Ranges

In [ ]:
numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()  # Identify fields currently stored as numeric.
for column in numeric_columns:  # Inspect every numeric field.
    print(f"\n--- {column} ---")  # Print the field name.
    print("Minimum:", df[column].min())  # Display the minimum value.
    print("Maximum:", df[column].max())  # Display the maximum value.
    print("Unique values:", df[column].nunique())  # Display the number of distinct values.

### Narration

This provides an initial validity/range check.

The documented `tenure` range is **0–72 months**. `SeniorCitizen` is represented as a binary 0/1 indicator.

This stage is not intended to replace EDA/outlier analysis. It is an initial structural sanity check.

## 18. Build the Raw-Data Quality Summary

In [ ]:
quality_summary = pd.DataFrame({  # Build a concise table of the main raw-data checks.
    "Check": ["Dataset shape", "Exact duplicate rows", "Duplicate customer IDs", "Pandas null values", "Whitespace-only TotalCharges", "TotalCharges dtype", "Target populated values"],  # Define the checks.
    "Observed": [f"{rows} rows × {columns} columns", duplicate_row_count, customer_id_duplicate_count, int(df.isnull().sum().sum()), int(totalcharges_blank_mask.sum()), str(df["TotalCharges"].dtype), int(df["Churn"].notna().sum())],  # Record observed results.
    "Status": ["PASS", "PASS", "PASS", "PASS*", "ACTION REQUIRED", "ACTION REQUIRED", "PASS"]  # Classify the findings for handoff.
})  # Complete the quality summary DataFrame.
display(quality_summary)  # Display the consolidated assessment.

### Narration

The raw-data baseline is:

| Check | Result |
|---|---|
| Dataset | 7,043 × 21 |
| Exact duplicates | 0 |
| Duplicate customer IDs | 0 |
| Formal pandas nulls | 0 |
| Whitespace-only `TotalCharges` | 11 |
| `TotalCharges` dtype | `object` |
| Target populated | 7,043 |

The asterisk on the null check is intentional: zero formal nulls do not mean zero data-quality issues. The whitespace anomaly demonstrates why content-based checks are necessary. fileciteturn2file1L237-L276

## 19. Shared Explanation of Important Customer Fields

In [ ]:
important_fields = {  # Define the agreed operational groups for the important customer fields.
    "Customer identity": ["customerID"],  # Identify the customer record.
    "Customer relationship": ["tenure", "Contract"],  # Describe duration and contractual commitment.
    "Financial behaviour": ["MonthlyCharges", "TotalCharges"],  # Describe recurring and cumulative charges.
    "Billing behaviour": ["PaymentMethod", "PaperlessBilling"],  # Describe payment and billing arrangements.
    "Core connectivity": ["PhoneService", "MultipleLines", "InternetService"],  # Describe primary services.
    "Ancillary digital services": ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"],  # Describe optional services.
    "Demographics and household": ["gender", "SeniorCitizen", "Partner", "Dependents"],  # Describe customer/household context.
    "Outcome": ["Churn"]  # Define the classification target.
}  # Finish the field-group dictionary.
for group, fields in important_fields.items():  # Iterate through each operational group.
    print(f"{group}: {', '.join(fields)}")  # Display the group and fields.

### Narration

The team agreed to use consistent business language for the major fields:

- **Customer identity:** `customerID` provides traceability but is not inherently behavioural.
- **Customer relationship:** `tenure` describes duration; `Contract` describes commitment.
- **Financial behaviour:** `MonthlyCharges` describes current recurring charges; `TotalCharges` describes cumulative charges.
- **Billing behaviour:** `PaymentMethod` and `PaperlessBilling` describe billing/payment arrangements.
- **Core connectivity:** `PhoneService`, `MultipleLines` and `InternetService` describe the main service setup.
- **Ancillary digital services:** the six add-on fields describe optional services and preserve structural “No service” categories.
- **Demographics/household:** `gender`, `SeniorCitizen`, `Partner` and `Dependents` describe profile context.
- **Outcome:** `Churn` is the label to be predicted.

These definitions are consistent with the team's completed field-definition deliverable.

## 20. Create the Technical Baseline for Handoff

In [ ]:
technical_baseline = {  # Store the key findings as a compact technical record.
    "rows": rows,  # Record the number of customer records.
    "columns": columns,  # Record the number of fields.
    "target": "Churn",  # Record the target field.
    "exact_duplicate_rows": duplicate_row_count,  # Record exact duplicate count.
    "duplicate_customer_ids": duplicate_ids,  # Record duplicate ID count.
    "pandas_null_values": int(df.isnull().sum().sum()),  # Record formal null count.
    "whitespace_totalcharges": blank_total,  # Record hidden blank count.
    "totalcharges_raw_dtype": str(df["TotalCharges"].dtype),  # Record the raw storage type.
    "blank_totalcharges_tenure_zero": int((df.loc[totalcharges_blank_mask, "tenure"] == 0).sum()),  # Record how many affected records have tenure zero.
    "blank_totalcharges_churn_no": int((df.loc[totalcharges_blank_mask, "Churn"] == "No").sum())  # Record how many affected records have Churn No.
}  # Finish the technical baseline.
display(pd.Series(technical_baseline, name="Observed value"))  # Display the baseline for review.

### Narration

This is the compact state of the raw dataset that downstream leads should use.

The critical handoff facts are the dataset size, target definition, absence of duplicate rows/IDs, absence of formal nulls, the 11 whitespace `TotalCharges` values, the raw text datatype, and the consistent tenure/churn pattern of those affected records.

## 21. Questions for Data Preparation

In [ ]:
handoff_questions = [  # Define unresolved questions that must be answered during preparation.
    "How should the 11 whitespace-only TotalCharges records be represented after numeric conversion?",  # Ask for the missing-value treatment.
    "Should customerID be retained for traceability but excluded from model features?",  # Confirm identifier treatment.
    "How will structural categories such as No internet service be encoded?",  # Confirm semantic preservation during encoding.
    "Which fields are genuinely available at prediction time?",  # Trigger leakage review.
    "How will training and prototype inference use the exact same preprocessing transformations?"  # Confirm reproducibility.
]  # Finish the handoff questions.
for number, question in enumerate(handoff_questions, start=1):  # Number the questions.
    print(f"{number}. {question}")  # Display each question.

### Narration

These questions deliberately belong to the next stage.

The project brief assigns Simon ownership of cleaning, datatypes, encoding, identifiers, splitting, leakage prevention and repeatable preprocessing. The whole team must still review major decisions before moving forward.

**Leadership principle:** identify the problem here, provide evidence, hand it over clearly, and do not silently make downstream decisions.

## 22. Data Understanding Stage Gate

In [ ]:
stage_gate = pd.DataFrame({  # Build a checklist for deciding whether Data Understanding is ready for team handoff.
    "Exit criterion": [  # Define the required conditions.
        "Dataset source and scope documented",  # Confirm provenance.
        "Rows, columns, target and schema explained",  # Confirm structure.
        "Important customer fields explained",  # Confirm shared field meanings.
        "Formal nulls checked",  # Confirm missing-value inspection.
        "Blank/whitespace values checked",  # Confirm hidden missing-value inspection.
        "Duplicate rows and IDs checked",  # Confirm duplication checks.
        "Datatype issues documented",  # Confirm datatype assessment.
        "Whole-team review completed",  # Confirm shared understanding.
        "Open preparation questions handed to Simon",  # Confirm downstream handoff.
    ],  # Finish the criteria.
    "Status": ["READY", "READY", "READY", "READY", "READY", "READY", "READY", "TO CONFIRM", "TO CONFIRM"]  # Record current gate status.
})  # Finish the stage-gate table.
display(stage_gate)  # Display the stage-gate checklist.

### Narration

The project brief requires the whole team to review and understand the output before moving forward. For dataset understanding, everyone should be able to explain one customer row, important fields and the target; for raw-data inspection, everyone reviews the initial quality issues before cleaning starts.

Therefore, a notebook is not “complete” merely because all cells execute. The team review and documented handoff are part of completion.

# 23. Final Data Understanding Summary

## Raw dataset baseline

- **7,043 customer records**
- **21 columns**
- **`Churn`** is the binary classification target.
- **0** exact duplicate rows.
- **0** duplicate customer IDs.
- **0** pandas-recognised null values.
- **11** whitespace-only `TotalCharges` records.
- `TotalCharges` is stored as text and requires explicit treatment before numerical modelling.
- All 11 affected `TotalCharges` records have `tenure = 0` and `Churn = No`.

## What this stage has accomplished

The notebook establishes the raw schema, target, field meanings and initial quality baseline. It also exposes the hidden whitespace anomaly that a basic `isnull()` check misses.

## What this stage has NOT done

It has not:

- cleaned the dataset,
- imputed the 11 `TotalCharges` values,
- encoded categorical variables,
- dropped `customerID`,
- created a train/test split,
- trained a model,
- or made the final preprocessing decision.

Those belong to later stages.

## Handoff

The Data Exploration Lead can now build business-question-driven EDA using the agreed field meanings and target definition.

The Data Preparation Lead can now define the cleaning and preprocessing plan, especially `TotalCharges`, identifier handling, categorical encoding and leakage checks.

The team's completed Data Understanding Deliverable states that this stage is complete when the team can explain the raw dataset without relying on the person who first inspected it.

### Project principle

**UNDERSTAND FIRST. BUILD SECOND. REVIEW TOGETHER.**
